<a href="https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/04_dspy_3_miprov2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" style="height: 28px; vertical-align: middle;"/></a>  **[▶️ Run this notebook in Colab](https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/04_dspy_3_miprov2.ipynb)**

# DSPy and MIPROv2 — prompts as parameters

> **Track 04 — Agents · Notebook 04 · Runtime: ≈30 s on CPU**
>
> **Prerequisites:** `04_agents/02` (structured outputs).
>
> **Papers:**
> - Khattab et al. 2023, *DSPy*
>   ([2310.03714](https://arxiv.org/abs/2310.03714)).
> - Opsahl-Ong et al. 2024, *Optimizing Instructions and Demonstrations
>   for Multi-Stage Language Model Programs* (MIPROv2,
>   [2406.11695](https://arxiv.org/abs/2406.11695)).

---

## What

DSPy treats the prompt text itself as a parameter. You declare a
**Signature** (`question -> answer`), DSPy compiles it into a prompt,
and an **optimiser** (MIPROv2 being the current default) searches
over prompt variations to improve a task metric.

MIPROv2 optimises two axes simultaneously:

1. **Instruction.** The natural-language preamble ("You are a helpful
   assistant. Answer concisely.").
2. **Demonstrations.** Which k-shot examples to include.

It uses Bayesian optimisation over a small pool of candidates
generated by a "teacher" LM, scored on a held-out dev set.

No `dspy` install. We simulate three instruction variants and three
demo pools, exhaustively score all 9 combinations on a synthetic
classification task, and verify the optimiser finds the best pair.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import numpy as np

from llm_systems_cookbook._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_04_dspy_3_miprov2")


## Task: classify sentences as positive / neutral / negative

A deterministic "LLM" scores each sentence with a rule: count
positive / negative lexicon words; the effective accuracy depends on
the instruction + demo pool given to it.


In [ ]:
set_seed(0)
rng = np.random.default_rng(0)

POS = {"love", "great", "excellent", "amazing", "wonderful", "fantastic", "good", "happy"}
NEG = {"terrible", "awful", "bad", "horrible", "worst", "hate", "disappointing"}


def true_label(text: str) -> str:
    toks = text.lower().split()
    p = sum(1 for t in toks if t in POS)
    n = sum(1 for t in toks if t in NEG)
    if p > n:
        return "positive"
    if n > p:
        return "negative"
    return "neutral"


# Mix of clearly-labeled sentences and tied sentences. Tied sentences
# are where the instruction + demo pool matter — the "specific" instruction
# handles them by falling back to neutral, while "misleading" biases them
# toward positive.
CLEAR_POS = ["I love this product, it is amazing.",
             "A wonderful fantastic experience, great value.",
             "Good food, happy staff, excellent night."]
CLEAR_NEG = ["The food was terrible and service was awful.",
             "Horrible and disappointing. The worst.",
             "Bad weather ruined the awful trip."]
# Tied: equal positive and negative words — these default to neutral under
# correct tie-break, but a biased instruction/demo pool mislabels them.
TIED_NEUTRAL = [
    "Great but also horrible.",
    "Love it, hate it.",
    "Good and bad in equal measure.",
    "Amazing but awful.",
    "Excellent and terrible in one package.",
    "Happy and disappointing ending.",
]
# No-sentiment neutrals: no positive/negative tokens at all.
PURE_NEUTRAL = ["It was ok, neither strong nor weak.",
                "Nothing to write home about, just average.",
                "Just there. Did the job. So-so."]

EXAMPLES = (CLEAR_POS + CLEAR_NEG + TIED_NEUTRAL + PURE_NEUTRAL) * 2
rng.shuffle(EXAMPLES)
# Ground truth for tied sentences is neutral (equal-count defaults that way).
LABELS = [true_label(t) for t in EXAMPLES]


## The LM stub

Given an instruction + demos + query, return a prediction. The stub's
"intelligence" is modulated by how aligned the instruction and demos
are with the task — a well-chosen pair boosts accuracy by 15-20 pp.


In [ ]:
INSTRUCTIONS = {
    "vague":    "Decide the sentiment of the sentence.",
    "specific": "Label each sentence with exactly one of: positive, neutral, negative. Count positive/negative words before deciding; if they tie, output neutral.",
    "misleading": "Always prefer the most intense emotion label. When unsure, output positive.",
}

DEMO_POOLS = {
    "balanced": [
        ("I love this.", "positive"),
        ("This is terrible.", "negative"),
        ("It is ok.", "neutral"),
    ],
    "skewed": [
        ("I love this.", "positive"),
        ("Great food.", "positive"),
        ("Wonderful.", "positive"),
    ],
    "empty": [],
}


_lm_rng = np.random.default_rng(0)


def lm_predict(text: str, instruction: str, demos: list[tuple[str, str]]) -> str:
    '''Simulated LM: the instruction governs the tie-break policy.

    - The *specific* instruction teaches the LM to output neutral on
      ties; accurate.
    - The *vague* instruction leaves ties to a random tie-break —
      models without clear guidance just guess.
    - The *misleading* instruction biases ties toward positive.
    Demo pools further shift the tie-break via implicit anchoring.
    '''
    toks = text.lower().split()
    p = sum(1 for t in toks if t in POS)
    n = sum(1 for t in toks if t in NEG)

    policy = "random"
    if "count positive/negative words" in instruction.lower():
        policy = "neutral"
    elif "prefer the most intense" in instruction.lower():
        policy = "positive"

    if demos:
        pos_frac = sum(1 for _, lbl in demos if lbl == "positive") / len(demos)
        neg_frac = sum(1 for _, lbl in demos if lbl == "negative") / len(demos)
        # Balanced demos + vague instruction = the balanced tie-break lands on neutral.
        if policy == "random" and pos_frac < 0.6 and neg_frac < 0.6 and pos_frac > 0 and neg_frac > 0:
            policy = "neutral"
        # Skewed demos override even the specific instruction slightly.
        if pos_frac > 0.6:
            policy = "positive"

    if p > n:
        return "positive"
    if n > p:
        return "negative"
    if policy == "neutral":
        return "neutral"
    if policy == "positive":
        return "positive"
    if policy == "negative":
        return "negative"
    # Random tie-break (degraded behaviour without guidance or demos).
    return str(_lm_rng.choice(["positive", "negative", "neutral"]))


def evaluate(instruction_name: str, demos_name: str) -> float:
    instr = INSTRUCTIONS[instruction_name]
    demos = DEMO_POOLS[demos_name]
    # Reset the LM's random tie-break stream for reproducibility across
    # (instruction, demos) evaluations.
    global _lm_rng
    _lm_rng = np.random.default_rng(0)
    correct = sum(lm_predict(t, instr, demos) == y for t, y in zip(EXAMPLES, LABELS, strict=True))
    return correct / len(EXAMPLES)


## MIPROv2-style optimisation

MIPROv2 uses Bayesian optimisation, but on a 9-cell grid exhaustive
search is identical to whatever the optimiser would do. We pretend
the optimiser evaluates 5 candidate pairs (sampled from the full
grid) and reports the best.

Real MIPROv2 uses TPE / bandit strategies because the candidate pool
is much larger (hundreds of instructions × many k-shot subsets).


In [ ]:
accuracy_grid: dict[tuple[str, str], float] = {}
for instr in INSTRUCTIONS:
    for demo in DEMO_POOLS:
        accuracy_grid[(instr, demo)] = evaluate(instr, demo)

print("full grid (instruction x demos):")
for instr in INSTRUCTIONS:
    row = "   ".join(f"{demo}={accuracy_grid[(instr, demo)]:.3f}" for demo in DEMO_POOLS)
    print(f"  {instr:<12}  {row}")

best_pair = max(accuracy_grid, key=accuracy_grid.get)
best_acc = accuracy_grid[best_pair]
baseline_acc = accuracy_grid[("vague", "empty")]
print(f"\nbaseline (vague + empty demos) accuracy = {baseline_acc:.3f}")
print(f"optimised ({best_pair[0]} + {best_pair[1]}) accuracy = {best_acc:.3f}")


def mipro_search(n_candidates: int = 5, seed: int = 0) -> tuple[tuple[str, str], float]:
    rng = np.random.default_rng(seed)
    pairs = list(accuracy_grid.keys())
    sample_idx = rng.choice(len(pairs), size=min(n_candidates, len(pairs)), replace=False)
    candidates = [pairs[int(i)] for i in sample_idx]
    scored = [(p, accuracy_grid[p]) for p in candidates]
    return max(scored, key=lambda x: x[1])


mipro_pair, mipro_acc = mipro_search(n_candidates=5)
print(f"MIPRO-search (5 candidates) found {mipro_pair} at {mipro_acc:.3f}")


In [ ]:
s.check(
    "optimiser_beats_baseline",
    lambda: best_acc > baseline_acc,
    msg=f"baseline={baseline_acc:.3f}  best={best_acc:.3f}",
)
s.check(
    "best_prompt_uses_informative_instruction_or_balanced_demos",
    lambda: best_pair[0] == "specific" or best_pair[1] == "balanced",
    msg=f"best = {best_pair}",
)
s.check(
    "mipro_within_2pp_of_optimal",
    lambda: mipro_acc >= best_acc - 0.02,
    msg=f"mipro={mipro_acc:.3f}  optimal={best_acc:.3f}",
)
s.check(
    "misleading_instruction_hurts",
    lambda: accuracy_grid[("misleading", "skewed")] < accuracy_grid[("specific", "balanced")],
    msg=f"misleading+skewed={accuracy_grid[('misleading', 'skewed')]:.3f}  "
        f"specific+balanced={accuracy_grid[('specific', 'balanced')]:.3f}",
)


## Exercises

1. **Real DSPy.** `pip install dspy-ai`, define a `dspy.Signature`, wrap
   your rule-based LM in `dspy.LM`, run `MIPROv2.compile(...)`. The
   API is similar to the stub above, just with async LM calls and
   actual bayesian search.
2. **Add a third axis.** Extend the grid with `temperature ∈ {0, 0.3,
   0.7}`. Grid search gets expensive; you need the TPE sampler.
3. **Bootstrap few-shot demonstrations.** Instead of a pre-written
   demo pool, have DSPy generate demos by running a teacher LM on the
   training set and keeping those whose predictions are correct.
   Measure how bootstrapped demos compare to human-written ones.

## References

- DSPy papers and documentation.
- *dspy.teleprompt.MIPROv2* source code — the instruction proposer
  and k-shot sampler are each ~200 lines and readable.


In [ ]:
s.summary()
s.save()
